In [ ]:
from pathlib import Path
import shutil

PROJECT_ROOT = Path.cwd() #project location
DATA_DIR = PROJECT_ROOT #where you want to download the zip file to

Project root: /Users/michaelsun/Documents/New project
Data dir: /Users/michaelsun/Documents/New project/data/raw
Free disk space: 566.54 GB
Note: the dataset zip is large (~14+ GB).


In [ ]:
import requests

#from paper
ZENODO_RECORD_ID = '10075352'
record_url = f'https://zenodo.org/api/records/{ZENODO_RECORD_ID}'

resp = requests.get(record_url, timeout=60)
resp.raise_for_status()
record = resp.json()

files = record.get('files', [])
if not files:
    raise RuntimeError('No files found in Zenodo record.')

print(f"Record title: {record.get('metadata', {}).get('title', 'N/A')}")
print(f"Number of files: {len(files)}")
for f in files:
    key = f.get('key', 'unknown')
    size_gb = f.get('size', 0) / (1024**3)
    print(f"- {key} ({size_gb:.2f} GB)")


Record title: magcil/guitar_style_dataset: Initial Release
Number of files: 1
- magcil/guitar_style_dataset-v1.0.0.zip (13.27 GB)


In [ ]:
import hashlib
from tqdm import tqdm
from pathlib import Path

# Select the zip file from the record
file_entry = None
for f in files:
    key = f.get('key', '')
    if key.endswith('.zip'):
        file_entry = f
        break

if file_entry is None:
    raise RuntimeError('Could not find a .zip file in this Zenodo record.')

file_key = file_entry['key']
download_url = file_entry.get('links', {}).get('content') or file_entry.get('links', {}).get('self')
expected_checksum = file_entry.get('checksum', '')

target_zip = DATA_DIR / Path(file_key).name
print(f'Will download: {file_key}')
print(f'Download URL: {download_url}')
print(f'Target file: {target_zip}')
print(f'Expected checksum: {expected_checksum}')

def download_with_resume(url: str, dest: Path, chunk_size: int = 8 * 1024 * 1024):
    dest.parent.mkdir(parents=True, exist_ok=True)
    existing = dest.stat().st_size if dest.exists() else 0

    headers = {}
    mode = 'wb'
    if existing > 0:
        headers['Range'] = f'bytes={existing}-'
        mode = 'ab'

    with requests.get(url, stream=True, headers=headers, timeout=120) as r:
        if r.status_code == 416:
            print('File already fully downloaded (server returned 416).')
            return
        if existing > 0 and r.status_code == 200:
            print('Server does not support resume for this request. Restarting download.')
            existing = 0
            mode = 'wb'
        r.raise_for_status()

        content_length = r.headers.get('Content-Length')
        total = int(content_length) + existing if content_length is not None else None

        with open(dest, mode) as f, tqdm(total=total, initial=existing, unit='B', unit_scale=True, desc=dest.name) as pbar:
            for chunk in r.iter_content(chunk_size=chunk_size):
                if chunk:
                    f.write(chunk)
                    pbar.update(len(chunk))

def md5sum(path: Path, chunk_size: int = 8 * 1024 * 1024) -> str:
    h = hashlib.md5()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(chunk_size), b''):
            h.update(chunk)
    return h.hexdigest()

download_with_resume(download_url, target_zip)
print(f'Download complete: {target_zip}')

if expected_checksum.startswith('md5:'):
    expected_md5 = expected_checksum.split(':', 1)[1]
    actual_md5 = md5sum(target_zip)
    print(f'Expected MD5: {expected_md5}')
    print(f'Actual   MD5: {actual_md5}')
    if expected_md5 != actual_md5:
        raise RuntimeError('Checksum mismatch. Download may be corrupted.')
    print('Checksum verified.')
else:
    print('No MD5 checksum provided by API in expected format; skipping verification.')


Will download: magcil/guitar_style_dataset-v1.0.0.zip
Download URL: https://zenodo.org/api/records/10075352/files/magcil/guitar_style_dataset-v1.0.0.zip/content
Target file: /Users/michaelsun/Documents/New project/data/raw/guitar_style_dataset-v1.0.0.zip
Expected checksum: md5:081794e5682b3e70e886dcd7344f3a8a


guitar_style_dataset-v1.0.0.zip: 100%|██████████| 14.3G/14.3G [23:50<00:00, 9.96MB/s]  


Download complete: /Users/michaelsun/Documents/New project/data/raw/guitar_style_dataset-v1.0.0.zip
Expected MD5: 081794e5682b3e70e886dcd7344f3a8a
Actual   MD5: 081794e5682b3e70e886dcd7344f3a8a
Checksum verified.


In [4]:
import zipfile

EXTRACT_AFTER_DOWNLOAD = True
extract_dir = DATA_DIR / 'guitar_style_dataset'

if EXTRACT_AFTER_DOWNLOAD:
    extract_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(target_zip, 'r') as zf:
        zf.extractall(extract_dir)
    print(f'Extracted to: {extract_dir}')
else:
    print('Extraction skipped. Set EXTRACT_AFTER_DOWNLOAD=True to extract.')


Extracted to: /Users/michaelsun/Documents/New project/data/raw/guitar_style_dataset
